# Author name match and merge

Disclaimer: this file is currently using data scraped around Oct-Dec 2025, and might not reflect the most recently updated author lists off either SciVal or OpenAlex. This will be updated with newer data ASAP. 

In [11]:
# import libraries 

import pandas as pd
import numpy as np
import json
import ast
from datetime import datetime

### 1. Read and clean OpenAlex author data for SFU

For the sake of simplicity, limit only to authors who have at least 1 publication since 2020. 

In [12]:
# read OpenAlex data 
oa_authors_raw = pd.read_csv("sfu_all_authors.csv")

oa_authors_raw.head()

,id,orcid,display_name,display_name_alternatives,works_count,cited_by_count,summary_stats,ids,affiliations,last_known_institutions,topics,topic_share,x_concepts,counts_by_year,works_api_url,updated_date,created_date
0,https://openalex.org/A5114378471,https://orcid.org/0000-0002-0721-8331,B. Liu,"['B. Liu', 'B L Liu', 'B Liu', 'B X Liu', 'B. ...",2266,110309,"{'2yr_mean_citedness': 6.5344311377245505, 'h_...",{'openalex': 'https://openalex.org/A5114378471...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I126520041', 'ro...","[{'id': 'https://openalex.org/T10048', 'displa...","[{'id': 'https://openalex.org/T10527', 'displa...","[{'id': '10138342', 'wikidata': 'https://www.w...","[{'year': 1965, 'works_count': 1, 'oa_works_co...",https://api.openalex.org/works?filter=author.i...,2025-11-02T23:13:26.583910,2016-06-24
1,https://openalex.org/A5019316470,https://orcid.org/0000-0002-7223-2965,M. C. Vetterli,"['M. C. Vetterli', 'A. Vgenopoulos', 'D. Ventu...",1859,105597,"{'2yr_mean_citedness': 5.203389830508475, 'h_i...",{'openalex': 'https://openalex.org/A5019316470...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I1304378839', 'r...","[{'id': 'https://openalex.org/T10048', 'displa...","[{'id': 'https://openalex.org/T10527', 'displa...","[{'id': '105702510', 'wikidata': 'https://www....","[{'year': 1983, 'works_count': 1, 'oa_works_co...",https://api.openalex.org/works?filter=author.i...,2025-11-02T23:13:26.583910,2016-06-24
2,https://openalex.org/A5077377484,https://orcid.org/0000-0003-0027-7969,J. Llorente Merino,"['J. Llorente Merino', 'J Llorente Merino', 'J...",1681,92792,"{'2yr_mean_citedness': 5.543181818181818, 'h_i...",{'openalex': 'https://openalex.org/A5077377484...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I18014758', 'ror...","[{'id': 'https://openalex.org/T10048', 'displa...","[{'id': 'https://openalex.org/T10527', 'displa...","[{'id': '109214941', 'wikidata': 'https://www....","[{'year': 2008, 'works_count': 1, 'oa_works_co...",https://api.openalex.org/works?filter=author.i...,2025-11-02T23:13:26.583910,2016-06-24
3,https://openalex.org/A5039614567,https://orcid.org/0000-0002-7807-7484,M. Danninger,"['M. Danninger', 'Danninger, M.', 'Danninger, ...",1430,62257,"{'2yr_mean_citedness': 5.130434782608695, 'h_i...",{'openalex': 'https://openalex.org/A5039614567...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I18014758', 'ror...","[{'id': 'https://openalex.org/T10048', 'displa...","[{'id': 'https://openalex.org/T10527', 'displa...","[{'id': '109214941', 'wikidata': 'https://www....","[{'year': 2007, 'works_count': 2, 'oa_works_co...",https://api.openalex.org/works?filter=author.i...,2025-11-01T23:18:31.786418,2016-06-24
4,https://openalex.org/A5100397026,https://orcid.org/0000-0003-1991-119X,Hao Zhang,"['Hao Zhang', 'H. Zhang', 'HAO ZHANG', 'HONG-J...",1370,63147,"{'2yr_mean_citedness': 2.3508771929824563, 'h_...",{'openalex': 'https://openalex.org/A5100397026...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I102345215', 'ro...","[{'id': 'https://openalex.org/T10627', 'displa...","[{'id': 'https://openalex.org/T10719', 'displa...","[{'id': '100082104', 'wikidata': 'https://www....","[{'year': 1993, 'works_count': 2, 'oa_works_co...",https://api.openalex.org/works?filter=author.i...,2025-11-02T23:13:26.583910,2016-06-24


In [31]:
# clean OpenAlex data 
oa_authors_clean = oa_authors_raw

oa_authors_clean['counts_clean'] = oa_authors_clean['counts_by_year'].apply(lambda x: ast.literal_eval(x))
oa_authors_clean['latest_year'] = oa_authors_clean['counts_clean'].apply(lambda x: x[-1]['year']) # -1 index grabs last year
oa_authors_clean['pubs_in_latest_yr'] = oa_authors_clean['counts_clean'].apply(lambda x: x[-1]['works_count'])

oa_authors_old = oa_authors_clean[oa_authors_clean['latest_published'] < 2020].reset_index().drop(columns=['index']) # keep for future reference if necessary 
oa_authors_clean = oa_authors_clean[oa_authors_clean['latest_published'] >= 2020].reset_index().drop(columns=['index']) # update clean table

print(oa_authors_clean['counts_clean'][1][-1]['year'])
print(oa_authors_clean['latest_published'].unique())
print(oa_authors_clean['pubs_in_latest_yr'].unique())
oa_authors_clean

2025
[2025 2022 2023 2024 2020 2021]
[113  90  64  92  44  27  10   7  22  41 167  78  35  37  12   1  45  14
  24  15  66   8  11   9  13  20  38   3  16   6  67  68   2  29  31   5
  77  17   4 214  18  23  57  70  19  21  61  26]


,id,orcid,display_name,display_name_alternatives,works_count,cited_by_count,summary_stats,ids,affiliations,last_known_institutions,...,topic_share,x_concepts,counts_by_year,works_api_url,updated_date,created_date,counts_clean,latest_published,latest_year,pubs_in_latest_yr
0,https://openalex.org/A5114378471,https://orcid.org/0000-0002-0721-8331,B. Liu,"['B. Liu', 'B L Liu', 'B Liu', 'B X Liu', 'B. ...",2266,110309,"{'2yr_mean_citedness': 6.5344311377245505, 'h_...",{'openalex': 'https://openalex.org/A5114378471...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I126520041', 'ro...",...,"[{'id': 'https://openalex.org/T10527', 'displa...","[{'id': '10138342', 'wikidata': 'https://www.w...","[{'year': 1965, 'works_count': 1, 'oa_works_co...",https://api.openalex.org/works?filter=author.i...,2025-11-02T23:13:26.583910,2016-06-24,"[{'year': 1965, 'works_count': 1, 'oa_works_co...",2025,2025,113
1,https://openalex.org/A5019316470,https://orcid.org/0000-0002-7223-2965,M. C. Vetterli,"['M. C. Vetterli', 'A. Vgenopoulos', 'D. Ventu...",1859,105597,"{'2yr_mean_citedness': 5.203389830508475, 'h_i...",{'openalex': 'https://openalex.org/A5019316470...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I1304378839', 'r...",...,"[{'id': 'https://openalex.org/T10527', 'displa...","[{'id': '105702510', 'wikidata': 'https://www....","[{'year': 1983, 'works_count': 1, 'oa_works_co...",https://api.openalex.org/works?filter=author.i...,2025-11-02T23:13:26.583910,2016-06-24,"[{'year': 1983, 'works_count': 1, 'oa_works_co...",2025,2025,90
2,https://openalex.org/A5077377484,https://orcid.org/0000-0003-0027-7969,J. Llorente Merino,"['J. Llorente Merino', 'J Llorente Merino', 'J...",1681,92792,"{'2yr_mean_citedness': 5.543181818181818, 'h_i...",{'openalex': 'https://openalex.org/A5077377484...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I18014758', 'ror...",...,"[{'id': 'https://openalex.org/T10527', 'displa...","[{'id': '109214941', 'wikidata': 'https://www....","[{'year': 2008, 'works_count': 1, 'oa_works_co...",https://api.openalex.org/works?filter=author.i...,2025-11-02T23:13:26.583910,2016-06-24,"[{'year': 2008, 'works_count': 1, 'oa_works_co...",2025,2025,64
3,https://openalex.org/A5039614567,https://orcid.org/0000-0002-7807-7484,M. Danninger,"['M. Danninger', 'Danninger, M.', 'Danninger, ...",1430,62257,"{'2yr_mean_citedness': 5.130434782608695, 'h_i...",{'openalex': 'https://openalex.org/A5039614567...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I18014758', 'ror...",...,"[{'id': 'https://openalex.org/T10527', 'displa...","[{'id': '109214941', 'wikidata': 'https://www....","[{'year': 2007, 'works_count': 2, 'oa_works_co...",https://api.openalex.org/works?filter=author.i...,2025-11-01T23:18:31.786418,2016-06-24,"[{'year': 2007, 'works_count': 2, 'oa_works_co...",2025,2025,92
4,https://openalex.org/A5100397026,https://orcid.org/0000-0003-1991-119X,Hao Zhang,"['Hao Zhang', 'H. Zhang', 'HAO ZHANG', 'HONG-J...",1370,63147,"{'2yr_mean_citedness': 2.3508771929824563, 'h_...",{'openalex': 'https://openalex.org/A5100397026...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I102345215', 'ro...",...,"[{'id': 'https://openalex.org/T10719', 'displa...","[{'id': '100082104', 'wikidata': 'https://www....","[{'year': 1993, 'works_count': 2, 'oa_works_co...",https://api.openalex.org/works?filter=author.i...,2025-11-02T23:13:26.583910,2016-06-24,"[{'year': 1993, 'works_count': 2, 'oa_works_co...",2025,2025,44
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4953,https://openalex.org/A5119544262,NaN,Dominic Jaworiski,['Dominic Jaworiski'],1,0,"{'2yr_mean_citedness': 0.0, 'h_index': 0, 'i10...",{'openalex': 'https://openalex.org/A5119544262...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I18014758', 'ror...",...,"[{'id': 'htt

### 2. Read and clean SciVal author data for SFU

Again, limit only to authors who have published at least once since 2020.

In [14]:
# read SciVal data